In [ ]:
"""
Live API test script — runs against the Docker backend at http://localhost:8000.
Run with: python test_apis.py
"""

import json
import sys
import requests

BASE = "http://localhost:8000"
PDF_PATH = "/Users/ram/Downloads/JD_-_Data_Science_Internship.pdf"
QUESTION = "What are the key responsibilities for this data science internship?"


def header(title: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


def show(resp: requests.Response) -> dict:
    try:
        body = resp.json()
        print(json.dumps(body, indent=2))
    except Exception:
        print(resp.text[:500])
        body = {}
    if not resp.ok:
        print(f"[ERROR] Status {resp.status_code}")
        sys.exit(1)
    return body


# ── 1. Health ──────────────────────────────────────────────────────────────────
header("1. Health Check")
show(requests.get(f"{BASE}/health"))


# ── 2. Upload PDF ──────────────────────────────────────────────────────────────
header("2. Upload PDF")
with open(PDF_PATH, "rb") as f:
    resp = requests.post(
        f"{BASE}/api/upload",
        files={"file": ("JD_-_Data_Science_Internship.pdf", f, "application/pdf")},
        data={"description": "Data Science Internship job description document"},
    )
upload_body = show(resp)
document_id = upload_body["document_id"]
print(f"\n  document_id = {document_id}")
print(f"  chunk_count = {upload_body['chunk_count']}")


# ── 3. Suggestions ─────────────────────────────────────────────────────────────
header("3. Suggested Questions")
show(requests.post(
    f"{BASE}/api/suggestions",
    json={"document_id": document_id},
))


# ── 4. Topics ──────────────────────────────────────────────────────────────────
header("4. Topic Extraction")
show(requests.post(
    f"{BASE}/api/topics",
    json={"document_id": document_id},
))


# ── 5. Query ───────────────────────────────────────────────────────────────────
header("5. RAG Query")
query_body = show(requests.post(
    f"{BASE}/api/query",
    json={"question": QUESTION, "document_id": document_id},
))
answer = query_body["answer"]
contexts = [s["content"] for s in query_body["sources"]]
print(f"\n  rewritten_query = {query_body['rewritten_query']}")
print(f"  sources returned = {len(contexts)}")


# ── 6. Evaluate ────────────────────────────────────────────────────────────────
header("6. RAGAS Evaluation")
show(requests.post(
    f"{BASE}/api/evaluate",
    json={
        "question": QUESTION,
        "answer": answer,
        "contexts": contexts,
        # Optional: add ground_truth to also get context_recall score
        # "ground_truth": "Expected answer here..."
    },
))


# ── 7. TTS ─────────────────────────────────────────────────────────────────────
header("7. Text-to-Speech")
tts_resp = requests.post(
    f"{BASE}/api/tts",
    json={"text": answer[:500], "voice": "nova"},
)
if tts_resp.ok:
    out_path = "/tmp/tts_output.wav"
    with open(out_path, "wb") as f:
        f.write(tts_resp.content)
    print(f"  WAV saved to {out_path} ({len(tts_resp.content)} bytes)")
    print(f"  Play with: afplay {out_path}")
else:
    show(tts_resp)

print("\n\nAll tests passed.\n")
